# 03 · Queries de verificación

Consultas analíticas sobre el modelo cargado en BigQuery, alineadas con las necesidades de negocio
del enunciado (ingresos, márgenes, segmentación, satisfacción, tendencias).

In [2]:
import os
from dotenv import load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account


load_dotenv()
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID",)
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

if CREDENTIALS_PATH:
    credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)
    client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
else:
    client = bigquery.Client(project=PROJECT_ID)

TABLE = lambda t: f"`{PROJECT_ID}.{DATASET_ID}.{t}`"
print(f"Conectado a {PROJECT_ID}.{DATASET_ID}")

c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Conectado a sqlproyect-507620.0


c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.cloud.bigquery once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery past that date.
  warnings.warn(message, FutureWarning)


## 1. Ingresos por mes

Solo se cuentan pagos `completed` (excluye reembolsos, fallidos y pendientes).

In [3]:
import db_dtypes

query_1 = f"""
SELECT
  FORMAT_TIMESTAMP("%Y-%m", payment_date) AS mes,
  ROUND(SUM(amount), 2) AS ingresos,
  COUNT(*) AS num_pagos
FROM {TABLE("payments")}
WHERE payment_status = "completed"
GROUP BY mes
ORDER BY mes
"""
client.query(query_1).to_dataframe()


c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,mes,ingresos,num_pagos
0,2023-04,3154.160000000,2
1,2023-05,7789.490000000,5
2,2023-06,5773.940000000,3
3,2023-07,16435.890000000,9
4,2023-08,5165.390000000,5
5,2023-09,12248.980000000,10
6,2023-10,17717.710000000,7
7,2023-11,11832.110000000,11
8,2023-12,17892.560000000,10
9,2024-01,19256.610000000,13


## 2. Top 10 productos más vendidos (por unidades)

In [4]:
query_2 = f"""
SELECT
  p.product_name,
  c.category_name,
  SUM(oi.quantity) AS unidades_vendidas,
  ROUND(SUM(oi.quantity * oi.unit_price - oi.discount_amount), 2) AS ingresos_generados
FROM {TABLE("order_items")} oi
JOIN {TABLE("products")} p ON p.product_id = oi.product_id
JOIN {TABLE("categories")} c ON c.category_id = p.category_id
GROUP BY p.product_name, c.category_name
ORDER BY unidades_vendidas DESC
LIMIT 10
"""
client.query(query_2).to_dataframe()

c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_name,category_name,unidades_vendidas,ingresos_generados
0,Novaphone X2 Pro,Smartphones,126,91558.810000000
1,LensCraft Action Cam,Fotografía,126,134907.410000000
2,AeroTech Pad Pro 12,Tablets,125,91468.320000000
3,Chronos Active Pro v2,Wearables,124,13527.730000000
4,Playtron Controller X,Gaming,123,9555.350000000
5,PixelPro Mirrorless M1,Fotografía,121,52215.170000000
6,Lumea Nova 5,Smartphones,120,144827.350000000
7,SoundWave Boom Mini v2,Audio,120,33356.200000000
8,PulseWear Band 6,Wearables,119,48382.330000000
9,BassPoint Pulse 2,Audio,118,28749.160000000


## 3. Clientes por país y canal de adquisición

In [5]:
query_3 = f"""
SELECT
  country,
  acquisition_channel,
  COUNT(*) AS num_clientes
FROM {TABLE("customers")}
GROUP BY country, acquisition_channel
ORDER BY country, num_clientes DESC
"""
client.query(query_3).to_dataframe()

c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,country,acquisition_channel,num_clientes
0,Alemania,organic,27
1,Alemania,social_media,22
2,Alemania,paid_ads,19
3,Alemania,referral,8
4,Alemania,email_marketing,7
5,Alemania,affiliate,4
6,Bélgica,organic,11
7,Bélgica,paid_ads,7
8,Bélgica,social_media,3
9,Bélgica,email_marketing,3


## 4. Tiempo medio de entrega por país (días entre `order_date` y `delivered_date`)

In [6]:
query_4 = f"""
SELECT
  shipping_country,
  ROUND(AVG(TIMESTAMP_DIFF(delivered_date, order_date, HOUR) / 24.0), 2) AS dias_medios_entrega,
  COUNT(*) AS pedidos_entregados
FROM {TABLE("orders")}
WHERE order_status = "delivered"
GROUP BY shipping_country
ORDER BY dias_medios_entrega
"""
client.query(query_4).to_dataframe()

c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,shipping_country,dias_medios_entrega,pedidos_entregados
0,Francia,5.06,193
1,Alemania,5.08,187
2,Italia,5.08,128
3,España,5.17,332
4,Polonia,5.20,46
5,Países Bajos,5.32,75
6,Portugal,5.37,123
7,Bélgica,5.43,50


## 5. Margen y rentabilidad por categoría

`margen = ingresos - coste`, sobre líneas de pedido efectivamente vendidas.

In [7]:
query_5 = f"""
SELECT
  c.category_name,
  ROUND(SUM(oi.quantity * oi.unit_price - oi.discount_amount), 2) AS ingresos,
  ROUND(SUM(oi.quantity * p.cost), 2) AS coste_total,
  ROUND(SUM(oi.quantity * oi.unit_price - oi.discount_amount) - SUM(oi.quantity * p.cost), 2) AS margen,
  ROUND(100 * (SUM(oi.quantity * oi.unit_price - oi.discount_amount) - SUM(oi.quantity * p.cost))
        / NULLIF(SUM(oi.quantity * oi.unit_price - oi.discount_amount), 0), 1) AS margen_pct
FROM {TABLE("order_items")} oi
JOIN {TABLE("products")} p ON p.product_id = oi.product_id
JOIN {TABLE("categories")} c ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY margen DESC
"""
client.query(query_5).to_dataframe()

c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category_name,ingresos,coste_total,margen,margen_pct
0,Fotografía,787756.100000000,469129.300000000,318626.800000000,40.400000000
1,Laptops,792257.510000000,484821.890000000,307435.620000000,38.800000000
2,Smartphones,590470.750000000,368047.830000000,222422.920000000,37.700000000
3,Tablets,331901.210000000,205302.180000000,126599.030000000,38.100000000
4,Gaming,272354.090000000,163360.950000000,108993.140000000,40.000000000
5,Wearables,238726.900000000,138778.830000000,99948.070000000,41.900000000
6,Audio,153199.660000000,86316.720000000,66882.940000000,43.700000000
7,Accesorios,34749.680000000,19938.350000000,14811.330000000,42.600000000


## 6. Satisfacción media (rating) por categoría

In [8]:
query_6 = f"""
SELECT
  c.category_name,
  ROUND(AVG(r.rating), 2) AS rating_medio,
  COUNT(*) AS num_valoraciones
FROM {TABLE("reviews")} r
JOIN {TABLE("order_items")} oi ON oi.order_item_id = r.order_item_id
JOIN {TABLE("products")} p ON p.product_id = oi.product_id
JOIN {TABLE("categories")} c ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY rating_medio DESC
"""
client.query(query_6).to_dataframe()

c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category_name,rating_medio,num_valoraciones
0,Smartphones,3.99,119
1,Gaming,3.92,116
2,Laptops,3.91,78
3,Fotografía,3.89,145
4,Audio,3.82,109
5,Wearables,3.81,126
6,Accesorios,3.77,113
7,Tablets,3.76,103


## 7. Clientes con mayor valor (top 10 por gasto total)

In [9]:
query_7 = f"""
SELECT
  cu.customer_id,
  cu.first_name,
  cu.last_name,
  cu.country,
  ROUND(SUM(p.amount), 2) AS gasto_total,
  COUNT(DISTINCT o.order_id) AS num_pedidos
FROM {TABLE("customers")} cu
JOIN {TABLE("orders")} o ON o.customer_id = cu.customer_id
JOIN {TABLE("payments")} p ON p.order_id = o.order_id AND p.payment_status = "completed"
GROUP BY cu.customer_id, cu.first_name, cu.last_name, cu.country
ORDER BY gasto_total DESC
LIMIT 10
"""
client.query(query_7).to_dataframe()

c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_id,first_name,last_name,country,gasto_total,num_pedidos
0,62,Tiburcio,Pellicer,España,23474.700000000,10
1,493,Dan,Bárcena,España,18927.740000000,5
2,223,Branka,Oderwald,Alemania,16333.930000000,5
3,366,Jaime,Ocaña,España,16318.500000000,7
4,160,Bernardino,Seco,España,15755.840000000,6
5,417,Margaux,Schneider,Francia,15537.870000000,7
6,74,Ricardo,Sanjuan,España,14731.860000000,5
7,324,Ada,Januszewicz,Polonia,14134.560000000,5
8,151,Caetana,Nascimento,Portugal,13981.810000000,7
9,252,Aleks,Siemieńczuk,Polonia,13955.830000000,6
